# PipeGuard: QLoRA distillation of gpt-oss-120b into a smaller reasoning model

The teacher (`openai/gpt-oss-120b` on Groq) investigated failing builds; its verified
**reasoning and answers** for the **Monitoring** and **Remediation** agents are the training data.
One setting picks the student:

| `MODEL` | Purpose |
|---|---|
| `qwen3-4b` | Usable locally: fits a 4 GB laptop GPU; exported to GGUF for Ollama |
| `gpt-oss-20b` | Strongest that trains on a free T4; same family and chat format as the teacher |
| `qwen3-14b` | Fallback if gpt-oss-20b runs out of memory |

**Kaggle setup** (or run it from the repo: `python -m training.kaggle_runner train --model <name>`)
1. *Settings → Accelerator*: **GPU T4 x2** (training uses one T4, 16 GB). *Internet*: **on**.
2. *Add Input*: a Kaggle dataset holding `train.jsonl` and `val.jsonl` from
   `backend/training/data/sft/` (produced by `python -m training.export_sft`).
3. Set `MODEL` below, then *Run All*. Run the notebook once per student.

In [ ]:
%%capture
!pip install --upgrade unsloth

In [ ]:
MODEL = "qwen3-4b"            # "qwen3-4b" | "gpt-oss-20b" | "qwen3-14b"

MODELS = {
    "qwen3-4b": {"name": "unsloth/Qwen3-4B", "family": "qwen3"},
    "gpt-oss-20b": {"name": "unsloth/gpt-oss-20b", "family": "gpt-oss"},
    "qwen3-14b": {"name": "unsloth/Qwen3-14B", "family": "qwen3"},
}
FAMILY = MODELS[MODEL]["family"]

import glob, os
# The attached dataset's mount path varies; locate train.jsonl wherever it is.
DATA_DIR = os.path.dirname(glob.glob("/kaggle/input/**/train.jsonl", recursive=True)[0])
MAX_SEQ_LENGTH = 4096        # examples with reasoning measure ~2,200 tokens; longer ones are dropped
EPOCHS = 2
MAX_EVAL = 20                # held-out examples per model; generation on a T4 is slow
MAX_NEW_TOKENS = 2500        # reasoning + answer
OUTPUT_DIR = f"/kaggle/working/pipeguard-{MODEL}-lora"

## 1. Load the student in 4-bit and attach LoRA adapters

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODELS[MODEL]["name"],
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
    dtype=None,
    full_finetuning=False,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=8,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## 2. Data

Each example is the exact production prompt and the teacher's verified reasoning + answer.
The teacher's reasoning is mapped to each family's own field: gpt-oss's *analysis* channel
(`thinking`) or Qwen3's `<think>` block (`reasoning_content`).

In [ ]:
import json
from collections import Counter
from datasets import Dataset

REASONING_FIELD = {"gpt-oss": "thinking", "qwen3": "reasoning_content"}[FAMILY]

def read_jsonl(path):
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def to_family(messages):
    converted = []
    for message in messages:
        message = dict(message)
        reasoning = message.pop("reasoning", None)
        if reasoning:
            message[REASONING_FIELD] = reasoning
        converted.append(message)
    return converted

def render(messages, add_generation_prompt=False):
    options = {"reasoning_effort": "medium"} if FAMILY == "gpt-oss" else {"enable_thinking": True}
    return tokenizer.apply_chat_template(
        to_family(messages), tokenize=False, add_generation_prompt=add_generation_prompt, **options,
    )

train_rows = read_jsonl(f"{DATA_DIR}/train.jsonl")
val_rows = read_jsonl(f"{DATA_DIR}/val.jsonl")

texts = [render(row["messages"]) for row in train_rows]
fits = [len(tokenizer(text)["input_ids"]) <= MAX_SEQ_LENGTH for text in texts]
train = Dataset.from_list([{"text": text} for text, ok in zip(texts, fits) if ok])

print("train:", len(train), "of", len(train_rows), Counter(r["meta"]["agent"] for r in train_rows))
print("val:  ", len(val_rows), Counter(r["meta"]["agent"] for r in val_rows))
print("with reasoning:", sum(bool(r["messages"][-1].get("reasoning")) for r in train_rows), "of", len(train_rows))
print(train[0]["text"][-1500:])

## 3. Train (loss on the assistant's reasoning and answer only)

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

MARKERS = {
    "gpt-oss": ("<|start|>user<|message|>", "<|start|>assistant<|channel|>"),
    "qwen3": ("<|im_start|>user\n", "<|im_start|>assistant\n"),
}[FAMILY]

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train,
    args=SFTConfig(
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        num_train_epochs=EPOCHS,
        learning_rate=2e-4,
        warmup_steps=5,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=5,
        seed=3407,
        output_dir="/kaggle/working/checkpoints",
        report_to="none",
    ),
)
# Mask the prompt: only the assistant turn (reasoning + answer) contributes to the loss.
trainer = train_on_responses_only(trainer, instruction_part=MARKERS[0], response_part=MARKERS[1])

In [ ]:
import torch

stats = trainer.train()
print(f"training time: {stats.metrics['train_runtime'] / 60:.1f} min")
print(f"peak GPU memory: {torch.cuda.max_memory_reserved() / 1024**3:.1f} GB")

## 4. Evaluate on held-out functions

The validation split holds out whole functions, so these are bugs in code the model never
saw. Each answer is checked the same way the generator labelled the teacher:

| Agent | `format_ok` | `grounded` |
|---|---|---|
| Monitoring | *Failed stage* + *Exact error* sections and `AGENT_CONFIDENCE` | names a failing test |
| Remediation | parsable steps JSON with at least 2 steps and `AGENT_CONFIDENCE` | names the buggy function |

`new_tokens` counts reasoning + answer: it is the cost and latency driver.

In [ ]:
import re
import time

STEP_START, STEP_END = "===STEPS_JSON_START===", "===STEPS_JSON_END==="

def final_answer(raw):
    if FAMILY == "gpt-oss":
        final = "<|channel|>final<|message|>"
        text = raw.split(final)[-1] if final in raw else raw
        for marker in ("<|return|>", "<|end|>"):
            text = text.split(marker)[0]
    else:
        text = raw.split("</think>")[-1].split("<|im_end|>")[0]
    return text.strip()

def check(agent, output, meta):
    has_confidence = re.search(r"AGENT_CONFIDENCE:\s*\d{1,3}", output) is not None
    if agent == "monitoring":
        tests = [test.split("::")[-1] for test in meta.get("failing_tests", [])]
        return {
            "format_ok": "**Failed stage**" in output and "**Exact error**" in output and has_confidence,
            "grounded": any(test in output for test in tests),
        }
    steps = []
    if STEP_START in output:
        block = output.split(STEP_START, 1)[1].split(STEP_END, 1)[0]
        try:
            steps = json.loads(block[block.find("["):block.rfind("]") + 1])
        except ValueError:
            steps = []
    return {
        "format_ok": len(steps) >= 2 and has_confidence,
        "grounded": meta.get("function", "") in output,
    }

def generate(messages):
    prompt = render(messages, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    started = time.perf_counter()
    output = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return {
        "text": final_answer(tokenizer.decode(new_tokens, skip_special_tokens=False)),
        "new_tokens": len(new_tokens),
        "seconds": time.perf_counter() - started,
    }

def evaluate(label, rows):
    results = []
    for row in rows:
        generated = generate(row["messages"][:-1])
        results.append({
            "model": label,
            "agent": row["meta"]["agent"],
            "group": row["meta"].get("group"),
            **check(row["meta"]["agent"], generated["text"], row["meta"]),
            "new_tokens": generated["new_tokens"],
            "seconds": round(generated["seconds"], 2),
            "output": generated["text"],
        })
        print(label, row["meta"]["agent"], {k: results[-1][k] for k in ("format_ok", "grounded", "new_tokens")})
    return results

In [ ]:
FastLanguageModel.for_inference(model)
eval_rows = val_rows[:MAX_EVAL]

results = evaluate(f"{MODEL} fine-tuned", eval_rows)
with model.disable_adapter():
    results += evaluate(f"{MODEL} base", eval_rows)

# Reference: the teacher's own (verified) reasoning + answers on the same examples.
for row in eval_rows:
    answer = row["messages"][-1]
    results.append({
        "model": "teacher (gpt-oss-120b)",
        "agent": row["meta"]["agent"],
        "group": row["meta"].get("group"),
        **check(row["meta"]["agent"], answer["content"], row["meta"]),
        "new_tokens": len(tokenizer((answer.get("reasoning") or "") + answer["content"])["input_ids"]),
        "seconds": None,
    })

In [ ]:
import pandas as pd

table = pd.DataFrame(results)
if table.empty:
    print("No validation examples: nothing to evaluate.")
    summary = pd.DataFrame()
else:
    summary = table.groupby(["model", "agent"]).agg(
        examples=("format_ok", "size"),
        format_ok=("format_ok", "mean"),
        grounded=("grounded", "mean"),
        mean_new_tokens=("new_tokens", "mean"),
        mean_seconds=("seconds", "mean"),
    ).round(2)
summary

## 5. Save the adapter and the results

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
table.to_json(f"/kaggle/working/eval_results_{MODEL}.jsonl", orient="records", lines=True)
summary.to_csv(f"/kaggle/working/eval_summary_{MODEL}.csv")
!cd /kaggle/working && zip -qr pipeguard-{MODEL}-lora.zip pipeguard-{MODEL}-lora
print(f"Download from the Output tab: pipeguard-{MODEL}-lora.zip, eval_summary_{MODEL}.csv")

## 6. Export for Ollama (local model only)

Merges the adapter into the model and writes a 4-bit GGUF file (~2.5 GB for Qwen3-4B) that
Ollama runs on a 4 GB laptop GPU. Building the converter takes several minutes.
After downloading, on the laptop:

```
ollama create pipeguard-qwen3-4b -f Modelfile     # Modelfile: FROM ./<the .gguf file>
```

In [ ]:
if MODEL == "qwen3-4b":
    model.save_pretrained_gguf(f"/kaggle/working/pipeguard-{MODEL}-gguf", tokenizer, quantization_method="q4_k_m")
    !ls -lh /kaggle/working/pipeguard-{MODEL}-gguf